# 13. Blessed 30M scaling runs (2 seeds: 123 / 456)

Same 30M setup as notebook 12, but runs **all three variants** for **two seeds**:
- `baseline` (micro_batch=32, grad_accum=2)
- `attnres` / Full (micro_batch=16, grad_accum=4)
- `block_attnres` / Block (micro_batch=16, grad_accum=4; `num_blocks` read from YAML)

Total planned runs: 6 (3 variants x 2 seeds).

In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/AtinChing/AttnResGPT-next-scale.git"
REPO_TARGET = Path("/content/AttnResGPT-next-scale")
DRIVE_ARTIFACT_FOLDER = "AttnResGPT-next-scale-artifacts"


def is_repo_root(path: Path) -> bool:
    return (path / "src" / "training" / "train.py").exists() and (path / "requirements.txt").exists()


def discover_repo() -> Path:
    for candidate in [REPO_TARGET, Path.cwd(), Path.cwd().parent]:
        if is_repo_root(candidate):
            return candidate.resolve()
    if not REPO_TARGET.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_TARGET)], check=True)
    return REPO_TARGET.resolve()


REPO = discover_repo()
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("repo:", REPO)
print("WANDB_API_KEY set =", bool(os.environ.get("WANDB_API_KEY")))

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_ARTIFACT_FOLDER
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("drive root:", DRIVE_ROOT)
print("artifacts layout:")
print("  checkpoints/", DRIVE_ROOT / "checkpoints")
print("  runs/       ", DRIVE_ROOT / "runs")
print("  logs/       ", DRIVE_ROOT / "logs")

In [ ]:
import torch

print("cuda_available =", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("device_name =", props.name)
    print("total_vram_gib =", round(props.total_memory / (1024 ** 3), 1))
    print("bf16_supported =", torch.cuda.is_bf16_supported())

In [ ]:
from src.utils.config import load_config
from src.models.attnres import build_model
from src.utils.logging import build_run_identity
from src.utils.runtime import count_parameters

CONFIG_PATH = "configs/fineweb_30m_blessed.yaml"
EFFECTIVE_BATCH_TOKENS = 65_536
SEEDS = [123, 456]

RUN_PLAN = [
    {
        "label": "baseline",
        "architecture": "baseline",
        "overrides": ["model.architecture=baseline", "model.attnres.enabled=false"],
        "notes": "Standard PreNorm residual GPT.",
    },
    {
        "label": "full_attnres",
        "architecture": "attnres",
        "overrides": [
            "model.architecture=attnres",
            "model.attnres.enabled=true",
            "model.attnres.mode=full",
        ],
        "notes": "Full depth-attention over all prior sublayer outputs.",
    },
    {
        "label": "block_attnres",
        "architecture": "block_attnres",
        "overrides": ["model.architecture=block_attnres"],
        "notes": "Block AttnRes (num_blocks read from YAML).",
    },
]

cfg = load_config(CONFIG_PATH, overrides=[f"experiment.seed={SEEDS[0]}"])

param_counts = {}
for plan in RUN_PLAN:
    variant_cfg = load_config(CONFIG_PATH, overrides=[f"experiment.seed={SEEDS[0]}", *plan["overrides"]])
    param_counts[plan["label"]] = count_parameters(build_model(variant_cfg.model))["total"]

run_names = {}
for seed in SEEDS:
    for plan in RUN_PLAN:
        variant_cfg = load_config(CONFIG_PATH, overrides=[f"experiment.seed={seed}", *plan["overrides"]])
        run_names[(seed, plan["label"])] = build_run_identity(variant_cfg).run_name

total_tokens = cfg.training.max_steps * EFFECTIVE_BATCH_TOKENS
print("=== Blessed 30M two-seed plan ===")
print(f"config file   : {CONFIG_PATH}")
print(f"seeds         : {SEEDS}")
print(f"model shape   : d_model={cfg.model.d_model} n_layers={cfg.model.n_layers} n_heads={cfg.model.n_heads} d_ff={cfg.model.d_ff}")
print(f"param counts  : baseline={param_counts['baseline']:,}  full={param_counts['full_attnres']:,}  block={param_counts['block_attnres']:,}")
print(f"horizon/run   : max_steps={cfg.training.max_steps} -> {total_tokens:,} tokens ({total_tokens/1e6:.1f}M)")
print(f"total budget  : {len(SEEDS) * len(RUN_PLAN) * total_tokens:,} tokens ({len(SEEDS) * len(RUN_PLAN) * total_tokens/1e6:.1f}M)")
print("\n=== run matrix ===")
for seed in SEEDS:
    print(f"seed={seed}")
    for plan in RUN_PLAN:
        print(f"  {plan['label']:14s} -> {run_names[(seed, plan['label'])]}")

In [ ]:
# Per-variant micro-batch settings matched to successful seed-1337 notebook 12 runs.
EFFECTIVE_BATCH_SEQUENCES = EFFECTIVE_BATCH_TOKENS // cfg.data.block_size  # 64

VARIANT_BATCH = {
    "baseline": (32, 2),       # seed 1337 baseline completed at mb=32
    "full_attnres": (16, 4),   # seed 1337 full completed at mb=16
    "block_attnres": (16, 4),  # seed 1337 block completed at mb=16
}

for label, (micro_batch, grad_accum) in VARIANT_BATCH.items():
    if EFFECTIVE_BATCH_SEQUENCES % micro_batch != 0:
        raise ValueError(f"{label}: micro_batch={micro_batch} must divide {EFFECTIVE_BATCH_SEQUENCES}")
    if micro_batch * grad_accum != EFFECTIVE_BATCH_SEQUENCES:
        raise ValueError(f"{label}: {micro_batch}x{grad_accum} != {EFFECTIVE_BATCH_SEQUENCES} sequences/step")

print("variant batch settings (65,536 tokens/step each):")
for label, (micro_batch, grad_accum) in VARIANT_BATCH.items():
    tokens = micro_batch * grad_accum * cfg.data.block_size
    print(f"  {label:14s} micro_batch={micro_batch:2d}  grad_accum={grad_accum}  -> {tokens:,} tokens/step")

In [ ]:
import shutil
from src.training.train import train_from_config

CONFIRM_LAUNCH = False
CLEAN_EXISTING_RUNS = False

SHARED_OVERRIDES = [
    f"logging.output_root={DRIVE_ROOT}",
    "logging.wandb.group=blessed_30m_scaling_seed_sweep",
]

print("launch plan:")
print("  CONFIRM_LAUNCH      =", CONFIRM_LAUNCH)
print("  CLEAN_EXISTING_RUNS =", CLEAN_EXISTING_RUNS)
for seed in SEEDS:
    for plan in RUN_PLAN:
        mb, ga = VARIANT_BATCH[plan["label"]]
        print(
            f"  seed={seed}  {plan['label']:14s}  mb={mb} ga={ga}  "
            f"run={run_names[(seed, plan['label'])]}"
        )

if not CONFIRM_LAUNCH:
    raise RuntimeError("Set CONFIRM_LAUNCH=True, then rerun this cell.")

RUN_SUMMARIES = []
suite_start = time.time()

for seed in SEEDS:
    for plan in RUN_PLAN:
        label = plan["label"]
        run_name = run_names[(seed, label)]
        micro_batch, grad_accum = VARIANT_BATCH[label]

        print("\n" + "=" * 72)
        print(f"LAUNCHING seed={seed} label={label} arch={plan['architecture']} mb={micro_batch} ga={grad_accum}")
        print("=" * 72)

        if CLEAN_EXISTING_RUNS:
            for sub in ("runs", "checkpoints"):
                path = DRIVE_ROOT / sub / run_name
                if path.exists():
                    print("removing", path)
                    shutil.rmtree(path)

        overrides = SHARED_OVERRIDES + [
            f"experiment.seed={seed}",
            f"data.batch_size={micro_batch}",
            f"data.eval_batch_size={micro_batch}",
            f"training.grad_accum_steps={grad_accum}",
            *plan["overrides"],
            f"logging.wandb.run_name={run_name}",
        ]
        launch_cfg = load_config(CONFIG_PATH, overrides=overrides)
        assert launch_cfg.training.eval_interval > 0
        assert launch_cfg.data.batch_size * launch_cfg.training.grad_accum_steps * launch_cfg.data.block_size == EFFECTIVE_BATCH_TOKENS

        run_start = time.time()
        summary = train_from_config(launch_cfg)
        run_hours = (time.time() - run_start) / 3600

        RUN_SUMMARIES.append({
            "seed": seed,
            "label": label,
            "micro_batch": micro_batch,
            "grad_accum": grad_accum,
            "run_name": summary.get("run_name"),
            "val_loss": summary.get("val_loss"),
            "best_val_loss": summary.get("best_val_loss"),
            "tokens_seen": summary.get("tokens_seen"),
            "wall_clock_hours": run_hours,
            "checkpoint_path": summary.get("checkpoint_path"),
            "wandb_url": summary.get("wandb_url"),
        })

        print(f"Finished seed={seed} label={label} in {run_hours:.2f} h")
        print("  val_loss      :", summary.get("val_loss"))
        print("  best_val_loss :", summary.get("best_val_loss"))
        print("  checkpoint    :", summary.get("checkpoint_path"))

_TRAINING_COMPLETED = True
suite_hours = (time.time() - suite_start) / 3600
print("\n" + "=" * 72)
print(f"ALL RUNS COMPLETE in {suite_hours:.2f} h")
for row in RUN_SUMMARIES:
    print(
        f"seed={row['seed']}  {row['label']:14s}  mb={row['micro_batch']}  "
        f"val={row['val_loss']}  best={row['best_val_loss']}  wall={row['wall_clock_hours']:.2f}h"
    )


## End the Colab session on completion

Unassigns the runtime after all runs finish so idle compute units aren't burned.


In [ ]:
UNASSIGN_ON_COMPLETE = True

training_completed = globals().get("_TRAINING_COMPLETED", False)

if UNASSIGN_ON_COMPLETE and training_completed:
    print("Training complete - ending Colab session to free the A100.")
    from google.colab import runtime
    runtime.unassign()
else:
    print(f"Not unassigning (UNASSIGN_ON_COMPLETE={UNASSIGN_ON_COMPLETE}, training_completed={training_completed}).")